## Übungsaufgabe: Fussballresultate

### 2. CSV-Datei in einen Pandas-Datenrahmen einlesen und ausgeben

In [ ]:
import sqlite3
import pandas as pd

df_results = pd.read_csv("results.csv")
df_results

### 3. Die benötigten Spalten auswählen

In [ ]:
columns = ["date", "home_team", "away_team", "home_score", "away_score", "city"]
df_football = df_results[columns]
df_football

### 4. Den Datenrahmen in der Datenbank speichern

Die Tabelle wird mit `if_exists="replace"` bei erneutem Ausführen aktualisiert.

In [ ]:
with sqlite3.connect("example.db") as connection:
    df_football.to_sql("football", connection, if_exists="replace", index=False)

### 5. Schweizer Siege in Bern mit einer Datenbankabfrage finden

Die Abfrage berücksichtigt, dass die Schweiz sowohl Heim- als auch Auswärtsteam sein kann.

In [ ]:
query = """
SELECT date, home_team, away_team, home_score, away_score, city
FROM football
WHERE city = ?
  AND (
    (home_team = ? AND home_score > away_score)
    OR (away_team = ? AND away_score > home_score)
  )
"""

with sqlite3.connect("example.db") as connection:
    swiss_wins_db = pd.read_sql_query(
        query,
        connection,
        params=("Berne", "Switzerland", "Switzerland"),
    )

swiss_wins_db

### 6. Dieselbe Abfrage nur mit dem Datenrahmen beantworten

In [ ]:
berne_switzerland_wins = df_football[
    (df_football["city"] == "Berne")
    & (
        (
            (df_football["home_team"] == "Switzerland")
            & (df_football["home_score"] > df_football["away_score"])
        )
        | (
            (df_football["away_team"] == "Switzerland")
            & (df_football["away_score"] > df_football["home_score"])
        )
    )
]

berne_switzerland_wins